# Cross-task semantic regression: picture vs auditory naming

Per-patient comparison of the PLS subspaces learned at each task's peak loose-semantic-category bin.

**Pipeline (per patient):**
1. Find peak `category_balanced_acc` bin in each task (from per_time_scores.csv).
2. Train fresh kernel-PLS at peak on ALL trials.
3. Compare 10D projection geometry: alignment index, principal angles, CCA.
4. Co-project 10D PLS scores into a common 2D CCA space; compare pre vs post CCA.
5. Cross-task decoding: pipeline_A applied to task B's data (at task-B peak), and vice versa.

Folder pair: `2026-04-08_..._kernel_pls_cosine_50ep` (picture) and `2026-05-06_..._auditory_naming_warp-linear_kernel_pls_cosine_50ep` (auditory).

**Memory note:** the picture-naming pkls are 0.7-2.6 GB each. Run on a machine with at least 16 GB RAM; otherwise process patients one at a time and call `gc.collect()` between.

In [ ]:
%matplotlib inline
import os, sys, gc
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

# Make project root importable
PROJECT_ROOT = Path.cwd().parents[1] if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from main.tests import cross_task_regression as ctr
print('module:', ctr.__file__)

## 1. Find peak bins per task (cheap — uses per_time_scores.csv only)

In [ ]:
PATIENT = 'AA'    # change to iterate; or set to None to use ctr.SHARED_PATIENTS
EMB = ctr.PEAK_EMBEDDING   # 'GloVe'

pic_scores = ctr.load_per_time_scores(ctr.PIC_RUN_DEFAULT, PATIENT)
aud_scores = ctr.load_per_time_scores(ctr.AUD_RUN_DEFAULT, PATIENT)
pic_peak, pic_val = ctr.find_peak_bin(pic_scores, EMB)
aud_peak, aud_val = ctr.find_peak_bin(aud_scores, EMB)
print(f'{PATIENT}: picture peak bin = {pic_peak} ({pic_val:.3f}); auditory peak bin = {aud_peak} ({aud_val:.3f})')

fig, ax = plt.subplots(figsize=(8, 3.5))
for label, df, color, peak in [('picture', pic_scores, 'C0', pic_peak),
                                ('auditory', aud_scores, 'C3', aud_peak)]:
    sub = df[df['embedding'] == EMB]
    t = sub['bin_index'].values * 0.1
    ax.plot(t, sub['category_balanced_acc'].values, label=label, color=color)
    ax.axvline(peak * 0.1, ls='--', alpha=0.4, color=color)
ax.set_xlabel('time relative to onset (s)'); ax.set_ylabel('category_balanced_acc')
ax.set_title(f'{PATIENT} | loose-category retrieval traces ({EMB})'); ax.legend(); plt.show()

## 2. Load pkl, train fresh PLS at each task's peak bin

In [ ]:
# Picture
pic_d = ctr.load_results_pkl(ctr.PIC_RUN_DEFAULT, PATIENT)
pic_reg = pic_d['regressors'][EMB]
X_pic = ctr.get_neural_at_bin(pic_reg, pic_peak)
y_pic = pic_reg.y
words_pic = np.asarray(pic_reg.labels).astype(str)
meta_pic = ctr.get_trial_metadata(pic_reg)
pipe_pic = ctr.fit_full_pls(X_pic, y_pic, random_state=42)
T_pic = ctr.transform_pls(pipe_pic, X_pic)
print('picture: X', X_pic.shape, 'y', y_pic.shape, 'PLS scores', T_pic.shape)

# Auditory
aud_d = ctr.load_results_pkl(ctr.AUD_RUN_DEFAULT, PATIENT)
aud_reg = aud_d['regressors'][EMB]
X_aud = ctr.get_neural_at_bin(aud_reg, aud_peak)
y_aud = aud_reg.y
words_aud = np.asarray(aud_reg.labels).astype(str)
meta_aud = ctr.get_trial_metadata(aud_reg)
pipe_aud = ctr.fit_full_pls(X_aud, y_aud, random_state=42)
T_aud = ctr.transform_pls(pipe_aud, X_aud)
print('auditory: X', X_aud.shape, 'y', y_aud.shape, 'PLS scores', T_aud.shape)

del pic_d; gc.collect()

## 3. Word-averaged matched matrices, alignment index, principal angles

In [ ]:
M_pic, M_aud, sw = ctr.matched_word_average(T_pic, words_pic, T_aud, words_aud)
print(f'shared words: {len(sw)}')
align = ctr.alignment_metrics(M_pic, M_aud)
print(f'alignment_index = {align["alignment_index"]:.3f}')
print(f'principal angles (deg): {np.round(align["principal_angles_deg"], 1)}')

fig, ax = plt.subplots(figsize=(5, 3))
ax.bar(np.arange(1, len(align['principal_angles_deg']) + 1), align['principal_angles_deg'],
       color=['#2ca02c' if a < 30 else '#ff7f0e' if a < 60 else '#d62728'
              for a in align['principal_angles_deg']])
ax.axhline(45, ls=':', color='grey'); ax.set_xlabel('PLS dim'); ax.set_ylabel('angle (deg)')
ax.set_title(f'Principal angles | align idx = {align["alignment_index"]:.2f}'); plt.show()

## 4. CCA alignment + quiver plot of PLS axes through CCA

In [ ]:
from sklearn.cross_decomposition import CCA
n_cca = min(ctr.N_PLS_COMPONENTS, len(sw) - 1)
cca, A_c, B_c, canon_corr = ctr.cca_align(M_pic, M_aud, n_components=n_cca)
print('canonical correlations:', np.round(canon_corr, 3))

cca2 = CCA(n_components=2, max_iter=2000)
Ac2, Bc2 = cca2.fit_transform(M_pic, M_aud)
ctr.plot_quiver_align(M_pic, M_aud, cca2, Ac2, Bc2, Path('quiver_align_inline.png'))
from IPython.display import Image, display
display(Image(filename='quiver_align_inline.png'))

## 5. Co-project trial-level PLS scores to 2D (pre vs post CCA)

In [ ]:
from sklearn.decomposition import PCA
T_pic_pre = PCA(n_components=2).fit_transform(T_pic)
T_aud_pre = PCA(n_components=2).fit_transform(T_aud)
T_pic_post = (T_pic - ctr._cca_x_mean(cca2)) @ cca2.x_rotations_[:, :2]
T_aud_post = (T_aud - ctr._cca_y_mean(cca2)) @ cca2.y_rotations_[:, :2]

ctr.plot_2d_trials(T_pic_pre, T_aud_pre, T_pic_post, T_aud_post,
                    meta_pic, meta_aud, Path('scatter_2d_inline.png'))
from IPython.display import Image, display
display(Image(filename='scatter_2d_inline.png'))

## 6. Cross-task decoding

In [ ]:
word_to_index_aud = {str(k): v for k, v in aud_reg.word_to_index.items()}
word_idx_to_cat_idx_aud = aud_reg.word_index_to_category_index
idx_to_cat_aud = aud_reg.index_to_category
word_to_index_pic = {str(k): v for k, v in pic_reg.word_to_index.items()}
word_idx_to_cat_idx_pic = pic_reg.word_index_to_category_index
idx_to_cat_pic = pic_reg.index_to_category

within_pic, cross_pic_to_aud = ctr.cross_task_decode(
    pipe_pic, X_pic, y_pic, words_pic, pipe_aud,
    X_aud, y_aud, words_aud,
    idx_to_cat_aud, word_idx_to_cat_idx_aud, word_to_index_aud,
)
within_aud, cross_aud_to_pic = ctr.cross_task_decode(
    pipe_aud, X_aud, y_aud, words_aud, pipe_pic,
    X_pic, y_pic, words_pic,
    idx_to_cat_pic, word_idx_to_cat_idx_pic, word_to_index_pic,
)
print('within_pic (full fit):    ', within_pic['category_balanced_acc'])
print('within_aud (full fit):    ', within_aud['category_balanced_acc'])
print('cross pic->aud:           ', cross_pic_to_aud['category_balanced_acc'])
print('cross aud->pic:           ', cross_aud_to_pic['category_balanced_acc'])
print('within_pic_holdout (CSV): ', pic_val)
print('within_aud_holdout (CSV): ', aud_val)

ctr.plot_cross_task_bars(
    within_pic['category_balanced_acc'], within_aud['category_balanced_acc'],
    cross_pic_to_aud['category_balanced_acc'], cross_aud_to_pic['category_balanced_acc'],
    pic_val, aud_val, Path('cross_task_bars_inline.png'),
)
from IPython.display import Image, display
display(Image(filename='cross_task_bars_inline.png'))

## 7. Run for all patients & generate cross-patient summary

```bash
python -m main.analysis.cross_task_regression
python -m main.report.cross_task_regression_report
```

The cell below runs the analysis inline.

In [ ]:
rows = []
for pat in ctr.SHARED_PATIENTS:
    try:
        rows.append(ctr.analyze_patient(pat, ctr.PIC_RUN_DEFAULT, ctr.AUD_RUN_DEFAULT, embedding=EMB))
    except Exception as e:
        print(f'  ERROR for {pat}: {e}')
summary = pd.DataFrame(rows)
print('\n\nCROSS-PATIENT SUMMARY:')
summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
x = np.arange(len(summary))
axes[0].bar(x - 0.2, summary['alignment_index'], 0.4, color='C0', label='alignment idx')
axes[0].bar(x + 0.2, summary['first_canon_corr'], 0.4, color='C1', label='first canon corr')
axes[0].set_xticks(x); axes[0].set_xticklabels(summary['patient']); axes[0].legend()
axes[0].set_title('Subspace alignment'); axes[0].set_ylim(0, 1.05)

axes[1].bar(x - 0.30, summary['within_pic_holdout'], 0.18, color='#9ecae1', label='pic holdout')
axes[1].bar(x - 0.10, summary['cross_aud_to_pic'], 0.18, color='#1f77b4', label='aud->pic')
axes[1].bar(x + 0.10, summary['within_aud_holdout'], 0.18, color='#fdae6b', label='aud holdout')
axes[1].bar(x + 0.30, summary['cross_pic_to_aud'], 0.18, color='#d62728', label='pic->aud')
axes[1].set_xticks(x); axes[1].set_xticklabels(summary['patient']); axes[1].legend(ncol=2, fontsize=8)
axes[1].set_title('Within (holdout) vs cross-task category retrieval'); axes[1].set_ylim(0)
fig.tight_layout(); plt.show()

### Generate the standalone HTML report

```bash
python -m main.report.cross_task_regression_report
```
Output: `semantic_regression_figures/cross_task_regression/cross_task_regression_report.html`